anomaly_context.json
        │
        ▼
Load JSON
        │
        ▼
Create Prompt
        │
        ▼
Groq API
        │
        ▼
Root Cause Analysis
        │
        ▼
Recommendations
        │
        ▼
Save Report

# AI-Powered Root Cause Analysis using Groq

This notebook performs automated Root Cause Analysis (RCA) on anomalous logs using a Large Language Model served through the Groq API.

For each detected anomaly, the surrounding context is analysed to determine:

- Probable Root Cause
- Severity
- Impact
- Recommended Fixes
- Confidence Score

The generated report can assist Site Reliability Engineers (SREs) in quickly identifying and resolving production issues.

In [1]:
!pip3 install groq
!pip3 install pandas 



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [2]:
pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from pathlib import Path

import pandas as pd

from groq import Groq

# Loading Context 

In [5]:
with open("../data/context/anomaly_context.json","r") as f:
    contexts = json.load(f)

print(f"Loaded {len(contexts)} anomaly contexts.")

Loaded 248 anomaly contexts.


In [6]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv(override=True)

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say SUCCESS"}
    ]
)

print(response.choices[0].message.content)

SUCCESS


In [7]:
import json

def build_prompt(context):

    return f"""
You are an expert Site Reliability Engineer (SRE).

Analyze the following anomalous log and provide a detailed Root Cause Analysis.

Return your answer in this JSON format:

{{
    "RootCause": "...",
    "Reason": "...",
    "Severity": "...",
    "Impact": "...",
    "Recommendations": [
        "...",
        "...",
        "..."
    ],
    "Confidence": "95%"
}}

Current Log

{context["CurrentLog"]}

Component

{context["Component"]}

Severity

{context["Severity"]}

Contains Error

{context["ContainsError"]}

Authentication Success

{context["AuthenticationSuccess"]}

Anomaly Score

{context["AnomalyScore"]}

Nearby Logs

{json.dumps(context["NearbyLogs"], indent=4)}

"""

In [ ]:
print(type(contexts[0]))
print(contexts[0].keys())

<class 'dict'>
dict_keys(['LogIndex', 'AnomalyScore', 'CurrentLog', 'Component', 'Severity', 'ContainsError', 'AuthenticationSuccess', 'NearbyLogs'])


In [8]:
# Sort anomalies by anomaly score (highest first)
contexts = sorted(
    contexts,
    key=lambda x: x["AnomalyScore"],
    reverse=True
)

TOP_K = 20

selected_contexts = contexts[:TOP_K]

print(f"Selected {len(selected_contexts)} highest-risk anomalies.")

Selected 20 highest-risk anomalies.


In [9]:
preview = pd.DataFrame(selected_contexts)[
    [
        "LogIndex",
        "Component",
        "Severity",
        "AnomalyScore"
    ]
]

preview.head(10)

,LogIndex,Component,Severity,AnomalyScore
0,4526,Globals.cpp,~CR~,3.947018
1,1101,Globals.cpp,~CR~,3.619014
2,4944,Globals.cpp,~CR~,3.566565
3,3605,Globals.cpp,~CR~,3.254536
4,2237,Globals.cpp,~CR~,3.162919
5,4160,Globals.cpp,~CR~,3.132577
6,362,Globals.cpp,~CR~,3.127324
7,454,Globals.cpp,~CR~,3.115484
8,1598,Globals.cpp,~CR~,3.111030
9,2219,Globals.cpp,~CR~,3.103915


In [10]:
rca_results = []

for i, context in enumerate(selected_contexts):

    print(f"Processing {i+1}/{TOP_K}")

    prompt = build_prompt(context)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    rca_results.append({
        "LogIndex": context["LogIndex"],
        "Response": response.choices[0].message.content
    })

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


In [11]:
parsed_results = []

for item in rca_results:

    try:
        result = json.loads(item["Response"])

        result["LogIndex"] = item["LogIndex"]

        parsed_results.append(result)

    except Exception as e:
        print(f"Skipped Log {item['LogIndex']}: {e}")

Skipped Log 4526: Expecting value: line 1 column 1 (char 0)
Skipped Log 1101: Expecting value: line 1 column 1 (char 0)
Skipped Log 4944: Expecting value: line 1 column 1 (char 0)
Skipped Log 3605: Expecting value: line 1 column 1 (char 0)
Skipped Log 4160: Expecting value: line 1 column 1 (char 0)
Skipped Log 362: Expecting value: line 1 column 1 (char 0)
Skipped Log 454: Expecting value: line 1 column 1 (char 0)
Skipped Log 1826: Expecting value: line 1 column 1 (char 0)
Skipped Log 4222: Expecting value: line 1 column 1 (char 0)
Skipped Log 1319: Expecting value: line 1 column 1 (char 0)
Skipped Log 2357: Expecting value: line 1 column 1 (char 0)
Skipped Log 874: Expecting value: line 1 column 1 (char 0)
Skipped Log 3093: Expecting value: line 1 column 1 (char 0)
Skipped Log 1186: Expecting value: line 1 column 1 (char 0)
Skipped Log 1648: Expecting value: line 1 column 1 (char 0)
Skipped Log 1094: Expecting value: line 1 column 1 (char 0)


In [12]:
rca_df = pd.DataFrame(parsed_results)

rca_df.head()

,RootCause,Reason,Severity,Impact,Recommendations,Confidence,LogIndex
0,High SMS traffic volume,The log indicates a high volume of SMS message...,Medium,The high SMS traffic volume may cause delays o...,[Monitor the SMS traffic volume and adjust the...,95%,2237
1,High volume of retry messages due to insuffici...,The logs indicate a high volume of retry messa...,Medium,The high volume of retry messages may cause in...,[Investigate the root cause of the insufficien...,95%,1598
2,High SMS traffic and potential misconfiguratio...,The logs indicate a high volume of SMS message...,Medium,The anomaly may cause delays or failures in SM...,[Investigate and optimize SMS routing configur...,95%,2219
3,High Traffic Volume,The logs indicate a high volume of SMS message...,Medium,The high traffic volume may cause delays or fa...,[Implement load balancing and scaling mechanis...,95%,342


In [13]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

rca_df.to_csv(
    "../outputs/root_cause_analysis.csv",
    index=False
)

print("Root Cause Analysis saved successfully.")

Root Cause Analysis saved successfully.


In [14]:
print("Average Confidence :", rca_df["Confidence"].mode()[0])

print("\nRoot Causes")

display(rca_df["RootCause"].value_counts())

print("\nSeverity Distribution")

display(rca_df["Severity"].value_counts())

Average Confidence : 95%

Root Causes


RootCause
High SMS traffic volume                                           1
High volume of retry messages due to insufficient balance         1
High SMS traffic and potential misconfiguration of SMS routing    1
High Traffic Volume                                               1
Name: count, dtype: int64


Severity Distribution


Severity
Medium    4
Name: count, dtype: int64